In [2]:
from google.colab import drive
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [3]:
drive.mount("/content/drive")
data = pd.read_csv("/content/drive/MyDrive/Colab_Notebooks/housing.csv")

data.info()

Mounted at /content/drive
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   price             545 non-null    int64 
 1   area              545 non-null    int64 
 2   bedrooms          545 non-null    int64 
 3   bathrooms         545 non-null    int64 
 4   stories           545 non-null    int64 
 5   mainroad          545 non-null    object
 6   guestroom         545 non-null    object
 7   basement          545 non-null    object
 8   hotwaterheating   545 non-null    object
 9   airconditioning   545 non-null    object
 10  parking           545 non-null    int64 
 11  prefarea          545 non-null    object
 12  furnishingstatus  545 non-null    object
dtypes: int64(6), object(7)
memory usage: 55.5+ KB


In [4]:
data_norm = pd.DataFrame({
    "price": data['price'],
    "area": data['area'],
    "bedrooms": data['bedrooms'],
    "bathrooms": data['bathrooms']
})
data_norm['area'] = (data_norm['area'] - data_norm['area'].min()) / (data_norm['area'].max() - data_norm['area'].min())
data_norm['price'] = (data_norm['price'] - data_norm['price'].min()) / (data_norm['price'].max() - data_norm['price'].min())
data_norm['bedrooms'] = (data_norm['bedrooms'] - data_norm['bedrooms'].min()) / (data_norm['bedrooms'].max() - data_norm['bedrooms'].min())
data_norm['bathrooms'] = (data_norm['bathrooms'] - data_norm['bathrooms'].min()) / (data_norm['bathrooms'].max() - data_norm['bathrooms'].min())
data_norm.head()

,price,area,bedrooms,bathrooms
0,1.000000,0.396564,0.6,0.333333
1,0.909091,0.502405,0.6,1.000000
2,0.909091,0.571134,0.4,0.333333
3,0.906061,0.402062,0.6,0.333333
4,0.836364,0.396564,0.6,0.000000


In [6]:

def regression(X: pd.DataFrame, w: pd.Series):
  return np.dot(X, w)

def loss_function(X: pd.DataFrame, y: pd.Series, w: pd.Series):
  return np.square(regression(X, w) - y).sum() / (2 * len(y))


def iteration(X: pd.DataFrame, y: pd.Series, w: pd.Series, learning_rate):
  m = len(y)
  grad = (X.T @ (regression(X, w) - y)) / m
  w -= learning_rate * grad
  return w

count = 0
def gradient(X, y, learning_rate, num_iter, eps):
  global count
  ones = np.ones((X.shape[0], 1))
  X = np.hstack((ones, X))

  w = np.zeros(X.shape[1])

  loss = loss_function(X, y, w)
  loss_history = [loss]

  for _ in range(num_iter):
    count += 1
    w = iteration(X, y, w, learning_rate)
    loss = loss_function(X, y, w)
    if abs(loss - loss_history[-1]) < eps:
      loss_history.append(loss)
      break

    loss_history.append(loss)

  return w, loss_history

X = pd.DataFrame({
        "area": data_norm['area'],
        "bedrooms": data_norm['bedrooms'],
        "bathrooms": data_norm['bathrooms']
    })
y = data_norm['price']

w, history_loss = gradient(X, y, 0.01, 1000, 0.0001)
print(f"Coefficients: {w}, MSE: {history_loss[-1]}")
print(count)

Coefficients: [0.14095698 0.04486501 0.0623708  0.02529905], MSE: 0.015115973541347165
90


In [14]:
regressor = LinearRegression().fit(X, y)
print(f"W0: {regressor.intercept_}\n W1-W3: {regressor.coef_}")
y_pred_train = regressor.predict(X)
mse_train = mean_squared_error(y, y_pred_train)
print(f"MSE: {mse_train}")

W0: 0.04282739976995409
 W1-W3: [0.47714269 0.17611257 0.36001286]
MSE: 0.01342681021702981


In [15]:
ones = np.ones((X.shape[0], 1))
X = np.hstack((ones, X))
w_analytical = np.linalg.inv(X.T @ X) @ X.T @ y

print(f'Analytical Solution: {w_analytical.round(4)}')

Analytical Solution: [0.0428 0.4771 0.1761 0.36  ]
